# A5. "50% 할인"은 진짜 파격 세일일까? — 할인 유형 진정성 검증

* 담당: 한예찬 · 분석 방향 문서의 **A5** (11/22 회의안은 "신작이 나오면 전작이 싸지나"였고, 팀장 제안으로 **"할인 진정성 검증: 상시 세일형 vs 희소 빅세일형"** 으로 바꿈)
* 한 줄 질문: **"지금 '50% 할인 중'이라는 표시, 진짜 파격 세일일까? 아니면 1년 내내 붙어 있는 상시 할인가일까?"**
* 새로 수집한 데이터는 없습니다. 수집기가 모아 정제·저장해 둔 **가격 이력(`price_history`)** 을 분석합니다.

### 데이터 처리 과정 (이 프로젝트의 수집 → 전달 → 정제 → 저장 → 활용 중 이 노트북이 맡는 부분)

| 단계 | 내용 | 이 노트북에서 |
| --- | --- | --- |
| 수집 | ITAD(PC 스토어 가격 이력) | 이미 수집기가 `raw.*`로 전송 |
| 전달·정제 | Kafka → 정제 프로그램: 중복 제거, **원화 환산**, **게임 ID 통합** (스토어 상품 → IGDB `game_id`) | 이미 `refined.price_history`로 정제됨 |
| 저장 | Kafka Connect → MariaDB `price_history` (**가격이 바뀐 때만** 한 줄씩 쌓임) | SQL로 읽기 전용 추출한 TSV를 불러옴 |
| 활용 | 게임마다 "얼마나 자주·깊게·오래 할인하는가" 지표 만들기 → 4유형 분류 → 유형별 실질 구매가 비교 → 서비스 제안 | 3~8장 |

### 분석 순서
1. 환경 설정 → 2. **가설 설정** → 3. 데이터 불러오기·점검 → 4. 전처리(지표 만들기) → 5. 탐색(EDA) → 6. **통계 검정** → 7. 의미 있는 결과인가? → 8. **인사이트·제안**

### 한눈에 보기 (2026-11-26까지의 데이터 기준)
1. **같은 할인율도 값어치가 다르다.** 최대 할인율은 84% vs 80%로 비슷한데, 1년 평균 구매가는 정가의 73% vs 80%였습니다.
2. **겜누리 화면의 최저가(전 스토어) 기준으로는 "할인 중"이 평상시 상태입니다.** 게임의 71%가 1년의 절반 넘게 할인 중입니다. (Steam만 보면 0%)
3. 유형 간 차이는 생각보다 작고(정가의 9~27% 할인), 4유형은 편의상의 구분입니다.
4. 제안: "진짜 딜" 표시는 할인율이 아니라 **이 게임의 1년 가격 분포에서 지금 가격의 위치**로 만듭니다.

> 이 노트북의 숫자는 모두 실행 결과로 계산됩니다. 위 요약과 각 "의미 판단"의 숫자는 2026-11-26 데이터로 실행한 값이고, 최종 결론은 8장에서 실제 값으로 자동 출력됩니다. **★** 는 발표에서 보여 줄 핵심 차트입니다.

### 준비 (한 번만)
```bash
pip install pandas numpy scipy matplotlib statsmodels jupyterlab
```
* DB 데이터 추출 (mariadb-re, 읽기 전용 계정). **A1 노트북(`01_A1_할인패턴예측`)과 같은 파일**을 씁니다.
```bash
mysql -u <읽기 계정> -p gamenuri -B --default-character-set=utf8mb4 < analysis/scripts/a1_extract.sql > analysis/data/a1_price_history.tsv
```
* 분석 대상 게임 목록은 `analysis/game_list/games_1000.csv` (Git에 있음)를 씁니다.
* 노트북은 `analysis/notebooks` 폴더에서 열어야 `../data` 경로가 맞습니다.

## 1. 환경 설정

In [ ]:
# 불필요한 경고 메시지를 숨겨 출력이 깔끔하게 보이도록 합니다.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 분석에 필요한 라이브러리를 불러옵니다.
# pandas/numpy: 데이터 처리, scipy: 통계 검정, matplotlib: 시각화
import os, json
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats
from statsmodels.stats.multitest import multipletests

%matplotlib inline

### 한글 폰트 설정

In [ ]:
# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트를 찾아 지정합니다. (Windows: 맑은 고딕, Mac: AppleGothic)
candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']
available = {f.name for f in fm.fontManager.ttflist}
font_name = next((c for c in candidates if c in available), None)

plt.rcParams['font.family'] = font_name or 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

BLUE, ORANGE, GRAY, GREEN, RED = '#2a78d6', '#eb6834', '#a3a29d', '#2f9e6e', '#d9453d'
print('사용 폰트:', font_name)

## 2. 가설 설정

데이터를 보기 **전에** 가설과 판정 기준을 먼저 정해 둡니다.

### 4가지 유형 (연간 할인 빈도 × 최대 할인율, 각 중앙값 기준으로 4분할)
| | 최대 할인율 높음 | 최대 할인율 낮음 |
| --- | --- | --- |
| **할인 자주** | **상시 세일형** — 자주, 깊게 | **미끼 할인형** — 자주, 얕게 |
| **할인 드물게** | **희소 빅세일형** — 드물지만 깊게 | **정가 고수형** — 거의 안 함 |

### 쓰는 지표 (최근 365일, 이력이 1년 이상인 유료 게임)
| 지표 | 뜻 |
| --- | --- |
| 연간 할인 빈도 | 할인(10% 이상)이 **시작된 횟수** |
| 최대 할인율 | 그 기간의 최대 할인율(%) |
| **실질 평균 구매가** | 1년 동안 매일 사는 사람이 평균 내는 가격 ÷ 정가 (1.0이면 정가) |
| 할인 일수 비율 | 1년 중 할인 중이었던 날의 비율 |
| 놓치면 기다리는 일수 | 할인이 아닌 날부터 **다음 할인이 시작되기까지** 평균 일수 |
| 할인 표시의 정확도 | 50% 이상 할인이 **표시된 날** 중, 가격이 1년 최저가의 105% 이내인 날의 비율 |

| # | 가설 (H1: 우리가 기대하는 것) | 귀무가설 (H0) | 통계 도구 | 실무 기준 ("의미 있다"의 조건) |
| --- | --- | --- | --- | --- |
| H1 | **유형에 따라 실질 평균 구매가가 다르다** | 4개 유형이 같다 | **Kruskal-Wallis**, 쌍별 Mann-Whitney (**Holm 보정**), 유형 간 격차의 부트스트랩 신뢰구간 | 가장 싼 유형과 가장 비싼 유형의 차이가 정가의 **10%p 이상** |
| H2 | **최대 할인율이 비슷해도** 구매가가 다르다 (상시 vs 희소) | 두 유형의 구매가가 같다 | **Mann-Whitney U**, 부트스트랩 신뢰구간 | 최대 할인율 차이 5%p 이내인데 구매가 차이는 **5%p 이상** |
| H3 | 희소 빅세일형은 **놓치면 더 오래 기다린다** | 상시 세일형과 같다 | **Mann-Whitney U**(단측), 중앙값 차이의 부트스트랩 신뢰구간 | 중앙값 차이 **7일 이상** |
| H4 | **자주 할인할수록 할인이 얕다** (미끼 할인 가설) | 관계가 없거나 반대다 | **Spearman 순위 상관**(단측), 부트스트랩 신뢰구간 | ρ ≤ **-0.2** |
| H5 | 유형별로 **"50% 이상 할인" 표시의 정확도**가 다르다 | 같다 | **Kruskal-Wallis** | 유형 간 정확도 차이 **10%p 이상** |

* **유의수준 α = 0.05.** 통계적으로 유의해도 효과가 작으면 "의미 없음"으로 판정합니다.
* **H1은 구조적으로 유의할 수밖에 없습니다.** 유형을 "빈도 × 깊이"로 정의했고 평균 구매가도 빈도와 깊이로 결정되기 때문입니다. 그래서 p값이 아니라 **차이의 크기**(실무 기준)로 판정합니다.
* **스토어 기준**: 유형은 **Steam** 가격으로 나눕니다. "상시 세일"은 한 스토어의 가격 정책이라, 여러 스토어의 최저가를 섞으면 키 판매점의 할인이 끼어듭니다. 겜누리 화면은 **전 스토어 최저가**를 보여 주므로, 그 기준도 함께 계산해 비교합니다.

### 분석 설정값 (한 곳에서 관리)

In [ ]:
# ---- 데이터 위치 ----
DATA_DIR = Path(os.environ.get('GAMENURI_DATA_DIR', '../data'))                  # analysis/data (Git에 올리지 않음)
RESULT_DIR = Path(os.environ.get('GAMENURI_RESULT_DIR', '../results/a5'))
GAME_LIST = Path(os.environ.get('GAMENURI_GAME_LIST', '../game_list/games_1000.csv'))
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 분석 정의 ----
SALE_MIN_PCT = 10           # 할인율 10% 이상인 날이 이어진 구간 = 할인 1회 (서비스 timing.py와 같은 기준)
YEAR = 365                  # 최근 365일, 이력이 1년 이상인 유료 게임만
NEAR_LOW = 1.05             # 1년 최저가의 105% 이내면 "최저가 근처"
MIN_TYPE_N = 10             # 유형 하나가 이보다 적으면 검정하지 않음(기준선 민감도에서)
TYPES = ['상시 세일형', '희소 빅세일형', '정가 고수형', '미끼 할인형']
COLORS = dict(zip(TYPES, [RED, BLUE, GRAY, ORANGE]))

# ---- 통계 ----
ALPHA = 0.05
PRACTICAL_PRICE = 0.10      # H1 실무 기준: 유형 간 평균 구매가 차이(최대-최소)가 정가의 10%p 이상
PRACTICAL_PRICE2 = 0.05     # H2 실무 기준: 구매가 차이 5%p 이상
SIMILAR_DISC = 5            # H2: 최대 할인율 차이가 5%p 이내면 "비슷하다"
PRACTICAL_WAIT = 7          # H3 실무 기준: 기다리는 일수 차이 7일 이상
PRACTICAL_RHO = -0.2        # H4 실무 기준: 상관계수 -0.2 이하
PRACTICAL_PREC = 0.10       # H5 실무 기준: 표시 정확도 차이 10%p 이상
N_BOOT, SEED = 1000, 42

R = {}                      # 최종 요약에 쓸 결과 저장소

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(RESULT_DIR / name, dpi=150, bbox_inches='tight')

## 3. 데이터 불러오기와 점검

In [ ]:
# mysql --batch로 뽑은 TSV를 읽습니다. 값이 없는 칸은 문자열 'NULL'로 나오므로 결측치로 바꿔 줍니다.
path = DATA_DIR / 'a1_price_history.tsv'
if not path.exists():
    raise FileNotFoundError(f'{path} 가 없습니다. 위 "준비" 칸의 mysql 명령으로 먼저 추출하세요.')
raw = pd.read_csv(path, sep='\t', na_values=['NULL'], parse_dates=['collected_at'])

games = pd.read_csv(GAME_LIST, encoding='utf-8-sig')                   # 분석 대상 게임 1,000개
games['name_ko'] = games['name_ko'].fillna(games['name'])
prices = raw[raw['game_id'].isin(games['game_id'])].copy()

print(f"가격 이력 {len(prices):,}행 · 게임 {prices['game_id'].nunique():,}개 · "
      f"{prices['collected_at'].min():%Y-%m-%d} ~ {prices['collected_at'].max():%Y-%m-%d}")
prices.head()

In [ ]:
# 데이터 타입과 결측치를 확인합니다. 분석 전에 반드시 구조를 점검하는 단계입니다.
prices.info()

In [ ]:
# 컬럼별 결측치. regular_krw(정가)·discount_pct는 스토어가 주지 않는 행에서 비어 있습니다.
# 정가가 없는 게임은 그 게임의 1년 최고가를 정가로 대신 씁니다. 할인율이 비면 0(= 할인 아님)으로 봅니다.
display(pd.DataFrame({'결측 수': prices.isnull().sum(), '결측률(%)': prices.isnull().mean() * 100}))

# 스토어별 행 수. 키 판매점이 많아 "모든 스토어의 최저가"는 거의 항상 할인 중입니다 (5장에서 확인).
prices['store'].value_counts().head(10).rename('행 수').to_frame().T

In [ ]:
# 숫자 컬럼의 기초 통계: 가격이 0 이하이거나 비정상적으로 큰 값이 없는지 확인합니다.
prices[['price_krw', 'regular_krw', 'discount_pct']].describe()

## 4. 전처리 — 게임별 지표 만들기

1. 게임마다 **일별 최저가·할인율**을 만들고, 가격이 안 바뀐 날도 직전 값으로 이어 채워 **달력 날짜 단위**로 펼칩니다. (그래야 "할인 일수 비율"과 "기다리는 일수"가 정확합니다)
2. 최근 365일의 지표를 계산합니다. **이력이 1년 미만이거나 무료인 게임은 뺍니다.**
3. **Steam만**, **전 스토어 최저가** 두 기준으로 각각 계산합니다.

In [ ]:
def daily_series(g):
    """게임 1개의 일별 최저가 (컬럼: day, price_krw, discount_pct). 스토어별 마지막 가격을 이어 채운 뒤 그날의 최저가만 남긴다."""
    g = g.assign(day=g['collected_at'].dt.normalize(), discount_pct=g['discount_pct'].fillna(0))
    last = g.sort_values('collected_at', kind='stable').groupby(['day', 'store', 'edition']).tail(1)
    price = last.pivot(index='day', columns=['store', 'edition'], values='price_krw').sort_index().sort_index(axis=1).ffill()
    disc = last.pivot(index='day', columns=['store', 'edition'], values='discount_pct').reindex_like(price).ffill()
    best = np.nanargmin(price.to_numpy(), axis=1)
    rows = np.arange(len(best))
    return pd.DataFrame({'day': price.index, 'price_krw': price.to_numpy()[rows, best],
                         'discount_pct': disc.to_numpy()[rows, best]})

END = prices['collected_at'].max().normalize()

def calendar_series(g):
    d = daily_series(g).set_index('day')[['price_krw', 'discount_pct']]
    return d.reindex(pd.date_range(d.index.min(), END)).ffill()            # 가격이 안 바뀐 날도 직전 값으로 채움

def game_metrics(g, regular, regular_mode='median'):
    cal = calendar_series(g)
    if cal.index.min() > END - pd.Timedelta(days=YEAR - 1):
        return None                                                        # 이력 1년 미만
    w = cal.loc[END - pd.Timedelta(days=YEAR - 1):END]
    price, disc = w['price_krw'].to_numpy(float), w['discount_pct'].to_numpy(float)
    if price.max() == 0:
        return None                                                        # 무료
    sale = disc >= SALE_MIN_PCT
    starts = np.flatnonzero(sale & ~np.r_[False, sale[:-1]])               # 할인이 시작된 날
    use_max = regular_mode == 'max' or not (regular and regular > 0)
    reg = price.max() if use_max else regular                              # 정가: 정가 중앙값(기본) 또는 1년 최고가
    off = np.flatnonzero(~sale)
    nxt = np.searchsorted(starts, off, side='right')
    has_next = nxt < len(starts)
    waits = starts[nxt[has_next]] - off[has_next]                          # 할인이 아닌 날 → 다음 할인 시작까지 일수
    big = disc >= 50
    return {'freq': len(starts), 'maxdisc': disc.max(), 'sale_share': sale.mean(),
            'avg_ratio': np.minimum(price / reg, 1.0).mean(),
            'wait_days': waits.mean() if len(waits) else np.nan,
            'deal_precision': (price[big] <= price.min() * NEAR_LOW).mean() if big.any() else np.nan}

def metrics_table(prices_sub, regular_mode='median'):
    regular = prices_sub.groupby('game_id')['regular_krw'].median()
    rows = {}
    for gid, g in prices_sub.groupby('game_id'):
        m = game_metrics(g, regular.get(gid), regular_mode)
        if m:
            rows[gid] = m
    return pd.DataFrame(rows).T.astype(float)

In [ ]:
prices_steam = prices[prices['store'] == 'steam']
m_steam_full = metrics_table(prices_steam)             # Steam 가격 기준
m_all_full = metrics_table(prices)                     # 겜누리 화면의 최저가(전 스토어) 기준
common = m_steam_full.index.intersection(m_all_full.index)
m_steam, m_all = m_steam_full.loc[common].copy(), m_all_full.loc[common].copy()

# 분석 표본이 되기까지 (단계별로 게임이 몇 개 남는가)
funnel = {'분석 대상 게임': len(games), '가격 이력이 있음': prices['game_id'].nunique(),
          '1년 이상 이력·유료 (전 스토어 기준)': len(m_all_full), '1년 이상 이력·유료 (Steam 기준도 만족) = 분석 대상': len(common)}
fn = pd.Series(funnel)
fig, ax = plt.subplots(figsize=(9, 3.4))
bars = ax.barh(fn.index[::-1], fn.values[::-1], color=BLUE)
for b_, v in zip(bars, fn.values[::-1]):
    ax.text(v, b_.get_y() + b_.get_height() / 2, f' {v:,}', va='center')
ax.set_title('분석 표본이 되기까지'); ax.set_xlabel('게임 수'); ax.set_xlim(0, fn.max() * 1.15)
savefig(fig, 'a5_1_funnel.png'); plt.show()
R['n_games'] = int(len(common))

### 의미 판단
- 1,000개 중 **908개(91%)** 가 분석 대상입니다. 빠진 92개는 대부분 **무료 게임**(약 75개)이고 나머지는 이력이 1년 미만인 게임(약 15개)입니다.
- 의미: A1(683개, 68%)보다 대상이 덜 줄었고, 빠진 이유가 분명합니다. 무료 게임은 "할인이 진짜인가"를 따질 대상이 아니라서 빠지는 것이 자연스럽습니다. 선택 편향을 크게 걱정하지 않아도 됩니다.

## 5. 탐색적 분석 (EDA)

### 5-1) 지표 요약 — 두 기준의 분포

In [ ]:
pd.concat({'Steam만': m_steam.describe().T[['mean', '25%', '50%', '75%']],
           '전 스토어 최저가': m_all.describe().T[['mean', '25%', '50%', '75%']]}, axis=1).round(2)

### 5-2) 먼저 확인 — "할인 중" 표시는 변별력이 있나? ★
거의 모든 게임이 거의 항상 할인 중이면, "할인 중"이라는 표시는 정보가 되지 못합니다.

In [ ]:
share_all = (m_all['sale_share'] >= .5).mean() * 100
share_steam = (m_steam['sale_share'] >= .5).mean() * 100
display(pd.DataFrame({
    '절반 이상의 날 할인 중인 게임(%)': [share_all, share_steam],
    '할인 일수 비율 중앙값(%)': [m_all['sale_share'].median() * 100, m_steam['sale_share'].median() * 100],
    '최대 할인율 50% 이상인 게임(%)': [(m_all['maxdisc'] >= 50).mean() * 100, (m_steam['maxdisc'] >= 50).mean() * 100],
}, index=['겜누리 최저가(전 스토어)', 'Steam만']).round(1))

fig, ax = plt.subplots(figsize=(9.5, 4))
ax.hist(m_all['sale_share'] * 100, bins=20, alpha=.65, label='전 스토어 최저가', color=RED)
ax.hist(m_steam['sale_share'] * 100, bins=20, alpha=.65, label='Steam만', color=BLUE)
ax.text(50, ax.get_ylim()[1] * .55, f'전 스토어 최저가: {share_all:.0f}%의 게임이\n1년의 절반 넘게 할인 중', fontsize=12, color='#b03a3a', fontweight='bold')
ax.set_title('★ 게임별 할인 일수 비율'); ax.set_xlabel('1년 중 할인 중이었던 날(%)'); ax.set_ylabel('게임 수'); ax.legend(loc='upper left')
savefig(fig, 'a5_2_sale_share.png'); plt.show()
R['share_half_sale'] = dict(all=float(share_all), steam=float(share_steam))

### 차트 해설
게임마다 "최근 1년 중 할인 중이었던 날"의 비율입니다. 빨강은 전 스토어 최저가 기준(겜누리 화면이 보여 주는 가격), 파랑은 Steam만입니다.

### 이 차트로 알 수 있는 것
- 빨강이 오른쪽 끝(거의 매일 할인)에 몰려 있으면, 키 판매점까지 포함한 최저가에서는 **"할인 중"이 평상시 상태**라는 뜻입니다. 이 기준으로는 할인율 숫자만으로 좋은 딜을 가리기 어렵습니다.
- 파랑(Steam)은 할인 중인 날이 일부라서 "세일 기간"이 실제로 구분됩니다. 그래서 **유형 분류는 Steam을 기준**으로 합니다.

### 의미 판단
- **겜누리 최저가(전 스토어) 기준으로는 게임의 70.8%가 1년의 절반 넘게 할인 중입니다**(할인 일수 비율 중앙값 83%). Steam만 보면 이 비율이 0%이고 중앙값이 29%입니다.
- 의미: 키 판매점 중 한 곳은 거의 항상 할인하기 때문에, 우리 화면의 가격에서 **"할인 중"은 평상시 상태**입니다. "할인 중" 표시나 할인율 순위만으로는 좋은 딜을 가려낼 수 없습니다.
- 서비스에 주는 뜻: "진짜 딜" 표시는 할인 여부가 아니라 **이 게임의 1년 가격 분포에서 지금 가격의 위치**로 만들어야 합니다. Steam은 세일 기간이 실제로 구분되므로 유형 분류의 기준으로 씁니다.

### 5-3) 유형 분류 — 연간 할인 빈도 × 최대 할인율

각 기준의 **중앙값**으로 4분할합니다. (기준값을 바꿔도 결론이 같은지는 7장에서 확인)

In [ ]:
def classify(t, f_thr, d_thr):
    hi_f, hi_d = t['freq'] >= f_thr, t['maxdisc'] >= d_thr
    return np.select([hi_f & hi_d, ~hi_f & hi_d, ~hi_f & ~hi_d, hi_f & ~hi_d], TYPES, default='')

thr = {}
for name, t in (('steam', m_steam), ('all', m_all)):
    thr[name] = (t['freq'].median(), t['maxdisc'].median())
    t['type'] = classify(t, *thr[name])

rng = np.random.default_rng(3)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
for ax, (name, t, title) in zip(axes, (('steam', m_steam, 'Steam만 (기준)'), ('all', m_all, '겜누리 최저가(전 스토어)'))):
    for ty in TYPES:
        s = t[t['type'] == ty]
        ax.scatter(s['freq'] + rng.uniform(-.3, .3, len(s)), s['maxdisc'] + rng.uniform(-1.5, 1.5, len(s)),
                   s=12, alpha=.5, color=COLORS[ty], label=f'{ty} ({len(s)})')
    f_thr, d_thr = thr[name]
    ax.axvline(f_thr - .5, color='k', lw=.8, ls='--'); ax.axhline(d_thr, color='k', lw=.8, ls='--')
    ax.set_title(title); ax.set_xlabel(f'연간 할인 빈도(회)  ·  기준 연 {f_thr:.0f}회 / {d_thr:.0f}%'); ax.legend(fontsize=8, loc='lower right')
axes[0].set_ylabel('최대 할인율(%)')
savefig(fig, 'a5_3_quadrant.png'); plt.show()
R['thresholds'] = {k: [float(v[0]), float(v[1])] for k, v in thr.items()}

### 차트 해설
점 하나가 게임 하나입니다(겹치지 않게 약간 흩뿌렸습니다). 점선이 기준선입니다.

### 이 차트로 알 수 있는 것
- 오른쪽 위 = 상시 세일형, 왼쪽 위 = 희소 빅세일형, 왼쪽 아래 = 정가 고수형, 오른쪽 아래 = 미끼 할인형입니다.
- 점이 경계에서 뚜렷하게 갈리지 않고 **연속적으로 퍼져 있으면** 4유형은 편의상의 구분입니다.
- 두 패널의 모양이 다르면 **스토어 기준에 따라 같은 게임의 유형이 달라진다**는 뜻입니다(7장에서 일치도를 잽니다).

### 의미 판단
- Steam 기준 유형별 게임 수는 상시 세일형 276개, 희소 빅세일형 189개, 정가 고수형 218개, 미끼 할인형 225개입니다(기준: 연 9회, 최대 할인율 75%). 전 스토어 기준은 240 / 267 / 175 / 226개입니다(기준: 연 6회, 75%).
- **점이 경계에서 갈리지 않고 연속적으로 퍼져 있습니다.** 4유형은 데이터가 자연스럽게 나눈 군집이 아니라 편의상의 구분입니다.
- **두 패널의 모양이 다릅니다.** 같은 게임이 어느 스토어를 기준으로 보느냐에 따라 다른 유형이 됩니다(7장에서 일치율 50%).
- 주의: 전 스토어 패널의 "희소 빅세일형"에는 1년 내내 할인 중인 게임이 섞여 있을 수 있습니다. 할인이 한 번 시작해서 끝나지 않으면 횟수는 적게 세기 때문입니다.

## 6. 가설 검정

공통 도구를 먼저 정의합니다. (판정 함수, 부트스트랩 신뢰구간)

In [ ]:
def verdict(p, effect_ok, direction_ok=True):
    if direction_ok and p < ALPHA and effect_ok:
        return '✅ 가설과 일치 — 통계적으로 유의하고 효과도 실무 기준을 넘음'
    if direction_ok and p < ALPHA:
        return '⚠️ 부분 일치 — 통계적으로는 유의하나 효과가 실무 기준에 못 미침'
    return '❌ 불일치 — 귀무가설을 기각하지 못함(차이가 있다고 말할 근거 부족) 또는 방향이 반대'

def mark(ok):
    return {True: '✅ 일치', False: '❌ 불일치', None: '검정 안 함'}[ok]

def mark_r(d, p_key='p'):
    # 효과가 실무 기준을 넘으면 일치, 통계적으로는 유의한데 효과가 기준에 못 미치면 부분 일치, 그 밖에는 불일치
    if d['ok']:
        return mark(True)
    return '⚠️ 부분 일치' if d.get(p_key, 1.0) < ALPHA else mark(False)

def boot(fn, *arrays, paired=False, B=N_BOOT, seed=SEED):
    """게임(행)을 다시 뽑아 통계량 fn의 95% 신뢰구간을 구합니다. 게임 한 개가 한 값이라 게임 단위 부트스트랩입니다.
    paired=True: 같은 게임의 값들을 함께 뽑음(상관계수처럼 짝지은 자료). False: 배열마다 따로 뽑음(서로 다른 두 집단)."""
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(B):
        if paired:
            idx = rng.integers(0, len(arrays[0]), len(arrays[0]))
            vals.append(fn(*[a[idx] for a in arrays]))
        else:
            vals.append(fn(*[a[rng.integers(0, len(a), len(a))] for a in arrays]))
    return np.percentile(vals, [2.5, 97.5])

def kruskal(t, col):
    groups = [t.loc[t['type'] == ty, col].dropna() for ty in TYPES]
    h, p = stats.kruskal(*groups)
    n = sum(len(g) for g in groups)
    return p, (h - len(TYPES) + 1) / (n - len(TYPES))                     # p값, 효과 크기 ε²

### 6-1) H1. 유형에 따라 실질 평균 구매가가 다른가? ★

> **주의**: 유형을 "빈도 × 깊이"로 정의했고 평균 구매가도 빈도와 깊이로 결정되는 값이라, 유의함은 **구조적으로 당연**합니다. 의미가 있는 것은 p값이 아니라 **차이의 크기**(정가 대비 몇 %p)입니다.

**Kruskal-Wallis**로 4개 유형 전체를 비교하고, **쌍별 Mann-Whitney U(Holm 보정)** 로 어느 유형끼리 다른지 봅니다. 신뢰구간은 각 유형의 평균 구매가를 게임 단위로 다시 뽑아 구합니다.

In [ ]:
p_h1, eps_h1 = kruskal(m_steam, 'avg_ratio')
by = {ty: m_steam.loc[m_steam['type'] == ty, 'avg_ratio'].to_numpy() for ty in TYPES}
means = pd.Series({ty: v.mean() for ty, v in by.items()}).reindex(TYPES)
cis = {ty: boot(np.mean, by[ty]) for ty in TYPES}
spread = float(means.max() - means.min())
rng = np.random.default_rng(SEED)
spread_boot = [np.ptp([rng.choice(by[ty], len(by[ty])).mean() for ty in TYPES]) for _ in range(N_BOOT)]
lo1, hi1 = np.percentile(spread_boot, [2.5, 97.5])
h1_ok = bool(p_h1 < ALPHA and spread >= PRACTICAL_PRICE and lo1 >= PRACTICAL_PRICE)

print(f'Kruskal-Wallis: p = {p_h1:.3g}, 효과 크기 ε² = {eps_h1:.2f}')
print(f'가장 싼 유형({means.idxmin()}) {means.min():.2f} ~ 가장 비싼 유형({means.idxmax()}) {means.max():.2f} → 차이 {spread:.2f} (95% 신뢰구간 {lo1:.2f} ~ {hi1:.2f})  ←  실무 기준 {PRACTICAL_PRICE}')
prs = list(combinations(range(len(TYPES)), 2))
raw_p = np.array([stats.mannwhitneyu(by[TYPES[i]], by[TYPES[j]]).pvalue for i, j in prs])
pair = pd.DataFrame({'비교': [f'{TYPES[i]} vs {TYPES[j]}' for i, j in prs],
                     '구매가 평균 차이': [round(by[TYPES[i]].mean() - by[TYPES[j]].mean(), 3) for i, j in prs],
                     'p(Holm 보정)': multipletests(raw_p, method='holm')[1]}).round(4)
display(pair)

summary = m_steam.groupby('type').agg(게임수=('freq', 'size'), 빈도중앙값=('freq', 'median'), 최대할인중앙값=('maxdisc', 'median'),
                                      할인일수비율=('sale_share', 'mean'), 평균구매가=('avg_ratio', 'mean'),
                                      기다리는일수=('wait_days', 'median'), 표시정확도=('deal_precision', 'mean')).reindex(TYPES)
display(summary.round(2))
R['h1'] = dict(p=float(p_h1), eps2=float(eps_h1), means={k: float(v) for k, v in means.items()}, spread=spread, ci=[float(lo1), float(hi1)], ok=h1_ok)
print('\nH1 판정:', verdict(p_h1, h1_ok))

fig, ax = plt.subplots(figsize=(8.5, 4))
ci_arr = np.array([cis[ty] for ty in TYPES])
ax.bar(TYPES, means.to_numpy(), color=[COLORS[t] for t in TYPES], yerr=[means.to_numpy() - ci_arr[:, 0], ci_arr[:, 1] - means.to_numpy()], capsize=4)
for i, v in enumerate(means.to_numpy()):
    ax.text(i, 0.04, f'{v:.2f}\n(정가의 {v * 100:.0f}%)', ha='center', color='white', fontweight='bold')
ax.axhline(1, color='k', lw=.8, ls='--')
ax.set(ylim=(0, 1.05), ylabel='정가 대비 실질 평균 구매가', title='★ 유형별로 실제로 얼마에 살 수 있나 (Steam, 95% 신뢰구간)')
savefig(fig, 'a5_4_h1.png'); plt.show()

### 차트 해설
정가를 1.0으로 했을 때, 1년 동안 매일 사는 사람이 평균적으로 내는 가격입니다. 오차 막대는 95% 신뢰구간입니다.

### 이 차트로 알 수 있는 것
- 막대가 낮을수록 평균적으로 싸게 살 수 있는 유형입니다. 1.0(점선)에 가까우면 사실상 정가에 사는 것과 같습니다.
- 가장 싼 유형과 가장 비싼 유형의 차이가 **0.1(정가의 10%p)** 을 넘으면 실무 기준을 충족합니다.
- 오차 막대가 겹치지 않아도 차이 자체가 작을 수 있습니다. 표본이 크면 신뢰구간은 좁아지므로 **막대 높이의 차이**를 봅니다.

### 의미 판단
- 평균 구매가는 상시 세일형 0.73(정가의 27% 할인), 희소 빅세일형 0.80(20%), 미끼 할인형 0.84(16%), 정가 고수형 0.91(9%)입니다. 가장 싼 유형과 가장 비싼 유형의 차이는 **0.18**(신뢰구간 0.17~0.19)로 실무 기준 0.1을 넘어 **✅ 가설과 일치**합니다.
- 의미: 가장 싸게 사는 상시 세일형도 평균적으로는 **정가의 73%** 를 냅니다. 네 유형 모두 정가의 73~91% 범위라서, 유형 간 차이는 생각보다 작습니다.

### 6-2) H2. 최대 할인율이 비슷해도 구매가가 다른가? (상시 세일형 vs 희소 빅세일형)

두 유형은 모두 "최대 할인율이 높은" 쪽입니다. 할인율 숫자가 비슷한데 실질 구매가가 다르면, 할인율만 보고는 구분할 수 없습니다.

In [ ]:
a = m_steam[m_steam['type'] == '상시 세일형']
b = m_steam[m_steam['type'] == '희소 빅세일형']
p_h2_disc = stats.mannwhitneyu(a['maxdisc'], b['maxdisc']).pvalue
p_h2 = stats.mannwhitneyu(a['avg_ratio'], b['avg_ratio']).pvalue
d_disc = float(b['maxdisc'].median() - a['maxdisc'].median())               # 최대 할인율 차이(%p, 중앙값)
d_price = float(a['avg_ratio'].mean() - b['avg_ratio'].mean())              # 구매가 차이 (음수 = 상시 세일형이 더 쌈)
lo2, hi2 = boot(lambda x, y: x.mean() - y.mean(), a['avg_ratio'].to_numpy(), b['avg_ratio'].to_numpy())
h2_ok = bool(p_h2 < ALPHA and abs(d_disc) <= SIMILAR_DISC and abs(d_price) >= PRACTICAL_PRICE2 and (hi2 < 0 or lo2 > 0))

print(f"최대 할인율(중앙값): 상시 {a['maxdisc'].median():.0f}% vs 희소 {b['maxdisc'].median():.0f}% (차이 {d_disc:+.0f}%p, p = {p_h2_disc:.3g})  ←  비슷함의 기준 {SIMILAR_DISC}%p 이내")
print(f"실질 평균 구매가  : 상시 {a['avg_ratio'].mean():.2f} vs 희소 {b['avg_ratio'].mean():.2f} (차이 {d_price:+.2f}, 95% 신뢰구간 {lo2:+.2f} ~ {hi2:+.2f}, Mann-Whitney p = {p_h2:.3g})  ←  실무 기준 {PRACTICAL_PRICE2}")
R['h2'] = dict(disc_a=float(a['maxdisc'].median()), disc_b=float(b['maxdisc'].median()), disc_diff=d_disc, price_a=float(a['avg_ratio'].mean()),
               price_b=float(b['avg_ratio'].mean()), price_diff=d_price, ci=[float(lo2), float(hi2)], p=float(p_h2), ok=h2_ok)
print('\nH2 판정:', verdict(p_h2, h2_ok))

### 의미 판단
- 상시 세일형과 희소 빅세일형은 **최대 할인율이 84% vs 80%로 비슷한데**(4%p 차이), 실질 평균 구매가는 **0.73 vs 0.80**(차이 −0.07, 신뢰구간 −0.09 ~ −0.06)입니다. **✅ 가설과 일치**합니다.
- 의미: **"80%대 할인"이라는 숫자만으로는 두 유형을 구분할 수 없습니다.** 상시 세일형은 자주 깊게 할인해서 평균 27%, 희소 빅세일형은 20% 싸게 삽니다. 할인율 순위만으로 게임을 추천하면 안 됩니다.

### 6-3) H3. 희소 빅세일형은 놓치면 더 오래 기다리는가?

"세일을 놓치면 몇 달 기다린다"는 주장을 확인합니다. **Mann-Whitney U**(단측)로 비교합니다.

In [ ]:
wa, wb = a['wait_days'].dropna().to_numpy(), b['wait_days'].dropna().to_numpy()
p_h3 = stats.mannwhitneyu(wb, wa, alternative='greater').pvalue
d_wait = float(np.median(wb) - np.median(wa))
lo3, hi3 = boot(lambda x, y: np.median(x) - np.median(y), wb, wa)
h3_ok = bool(p_h3 < ALPHA and d_wait >= PRACTICAL_WAIT and lo3 > 0)
print(f'놓치면 기다리는 일수(중앙값): 희소 빅세일형 {np.median(wb):.1f}일 vs 상시 세일형 {np.median(wa):.1f}일 (차이 {d_wait:+.1f}일, 95% 신뢰구간 {lo3:+.1f} ~ {hi3:+.1f}일, p = {p_h3:.3g})  ←  실무 기준 {PRACTICAL_WAIT}일')
R['h3'] = dict(wait_b=float(np.median(wb)), wait_a=float(np.median(wa)), diff=d_wait, ci=[float(lo3), float(hi3)], p=float(p_h3), ok=h3_ok)
print('\nH3 판정:', verdict(p_h3, h3_ok, direction_ok=d_wait > 0))

### 의미 판단
- 세일을 놓치면 다음 할인이 시작될 때까지 희소 빅세일형은 **25.5일**, 상시 세일형은 **15.8일**을 기다립니다(차이 +9.7일, 신뢰구간 +8.0 ~ +10.7일). **✅ 가설과 일치**합니다.
- 의미: "세일을 놓치면 오래 기다린다"는 말은 맞지만 몇 달이 아니라 **몇 주**(약 3.6주 vs 2.3주) 수준입니다.
- 주의: 7장에서 분류 기준선에 따라 이 차이가 5.8~22.4일로 달라집니다. 즉 "희소 빅세일형은 더 오래 기다린다"는 방향은 유지되지만 크기는 분류 기준에 따라 크게 달라지므로, 이를 근거로 한 서비스 제안은 약하게만 씁니다.

### 6-4) H4. 자주 할인할수록 할인이 얕은가? (미끼 할인 가설)

가설이 맞다면 할인 빈도와 최대 할인율은 **음의 상관**이어야 합니다. **Spearman 순위 상관**(단측)을 쓰고, 두 스토어 기준으로 각각 확인합니다.

In [ ]:
rho_s = stats.spearmanr(m_steam['freq'], m_steam['maxdisc'], alternative='less')
rho_a = stats.spearmanr(m_all['freq'], m_all['maxdisc'], alternative='less')
lo4, hi4 = boot(lambda x, y: stats.spearmanr(x, y).statistic, m_steam['freq'].to_numpy(), m_steam['maxdisc'].to_numpy(), paired=True)
h4_ok = bool(rho_s.pvalue < ALPHA and rho_s.statistic <= PRACTICAL_RHO)
print(f'Steam    : ρ = {rho_s.statistic:+.2f} (95% 신뢰구간 {lo4:+.2f} ~ {hi4:+.2f}), ρ<0 단측 p = {rho_s.pvalue:.3g}  ←  실무 기준 ρ ≤ {PRACTICAL_RHO}')
print(f'전 스토어: ρ = {rho_a.statistic:+.2f}, ρ<0 단측 p = {rho_a.pvalue:.3g}')
print('※ 전 스토어 기준의 음의 상관(약함)은 원인을 이 분석으로 확인하지 못했습니다. 1년 내내 할인 중인 게임은 할인 횟수가 적게 세어지는 계산 특성이 영향을 줬을 가능성이 있습니다.')
R['h4'] = dict(rho_steam=float(rho_s.statistic), ci=[float(lo4), float(hi4)], p_steam=float(rho_s.pvalue), rho_all=float(rho_a.statistic), p_all=float(rho_a.pvalue), ok=h4_ok)
print('\nH4 판정:', verdict(rho_s.pvalue, h4_ok, direction_ok=rho_s.statistic < 0))

### 의미 판단
- Steam에서 상관은 **+0.23**(신뢰구간 +0.16 ~ +0.29)입니다. 음의 상관이어야 "자주 할인할수록 얕다"가 성립하는데 **오히려 자주 할인하는 게임이 더 깊게 할인합니다.** **❌ 불일치**입니다.
- 전 스토어에서는 −0.13으로 약한 음의 상관이 나오지만 실무 기준(−0.2)에 못 미칩니다. 원인은 이 분석으로 확인하지 못했고, 1년 내내 할인 중인 게임의 할인 횟수가 적게 세어지는 계산 특성이 영향을 줬을 가능성이 있습니다.
- 의미: **"미끼 할인(잦고 얕은 할인)이 문제"라는 가설은 이 데이터로 뒷받침되지 않습니다.** 퍼블리셔에게 "미끼 할인을 줄이라"고 제안할 근거가 없고, 이 분석은 할인이 구매 행동에 미치는 영향을 보지 못합니다.

### 6-5) H5. 유형별로 "50% 이상 할인" 표시의 정확도가 다른가?

50% 이상 할인이 **표시된 날** 중, 그 가격이 **1년 최저가 근처(105% 이내)** 인 날의 비율입니다. 낮을수록 "50% OFF"를 보고 사도 **더 싸게 살 기회가 남아 있었다**는 뜻입니다. **Kruskal-Wallis**로 비교합니다.

In [ ]:
p_h5, eps_h5 = kruskal(m_steam, 'deal_precision')
prec = m_steam.groupby('type')['deal_precision'].agg(['count', 'mean', 'median']).reindex(TYPES)
display(prec.round(2).rename(columns={'count': '게임수(해당 날 있는)'}))
spread5 = float(prec['mean'].max() - prec['mean'].min())
h5_ok = bool(p_h5 < ALPHA and spread5 >= PRACTICAL_PREC)
print(f"Kruskal-Wallis: p = {p_h5:.3g}, 효과 크기 ε² = {eps_h5:.2f} · 유형 간 정확도 차이 {spread5:.2f} (최소 {prec['mean'].idxmin()} {prec['mean'].min():.2f} ~ 최대 {prec['mean'].idxmax()} {prec['mean'].max():.2f})  ←  실무 기준 {PRACTICAL_PREC}")
R['h5'] = dict(p=float(p_h5), eps2=float(eps_h5), means={k: float(v) for k, v in prec['mean'].items()}, spread=spread5, ok=h5_ok)
print('\nH5 판정:', verdict(p_h5, h5_ok))

### 의미 판단
- "50% 이상 할인" 표시가 뜬 날 중 가격이 1년 최저가 근처인 비율은 상시 세일형 **0.50**, 희소 빅세일형 0.62, 미끼 할인형 0.65, 정가 고수형 0.77입니다. 유형 간 차이는 0.27로 실무 기준 0.1을 넘어 **✅ 가설과 일치**합니다.
- 의미: **상시 세일형에서는 "50% 이상 할인" 표시가 떠도 절반은 1년 최저가보다 5% 넘게 비쌉니다.** 더 싸질 기회가 남아 있었다는 뜻입니다. 정가 고수형은 50% 할인이 곧 최저가인 경우가 많습니다. 그래서 할인율 표시만으로는 "지금이 사기 좋은 때"를 판단할 수 없습니다.
- 주의: 효과 크기(ε²)는 0.06으로 작고, 정가 고수형은 해당하는 날이 있는 게임이 127개뿐이라 표본이 작습니다.

## 7. 이 결과는 의미가 있는가?

p값만으로 결론을 내리지 않고 네 가지를 함께 봅니다.

1. **통계적 유의성**: p < 0.05
2. **효과 크기**: 실무 기준(H1 정가의 10%p, H2 5%p, H3 7일, H4 ρ -0.2, H5 10%p)을 넘는가
3. **신뢰구간**: 게임 단위 95% 신뢰구간이 0을 포함하지 않는가
4. **강건성**: 분석 조건(분류 기준선, 정가 정의, 스토어 기준)을 바꿔도 결론이 유지되는가

In [ ]:
def type_stats(t, f_thr, d_thr):
    tt = t.assign(type=classify(t, f_thr, d_thr))
    cnt = tt['type'].value_counts()
    if min(cnt.get(ty, 0) for ty in TYPES) < MIN_TYPE_N:
        return None
    x, y = tt[tt['type'] == TYPES[0]], tt[tt['type'] == TYPES[1]]
    means_ = tt.groupby('type')['avg_ratio'].mean()
    prec_ = tt.groupby('type')['deal_precision'].mean()
    return {'가장 작은 유형(게임수)': int(cnt.min()),
            'H1 구매가 차이(최대-최소)': means_.max() - means_.min(),
            'H2 상시-희소 구매가': x['avg_ratio'].mean() - y['avg_ratio'].mean(),
            'H3 희소-상시 기다림(일)': y['wait_days'].median() - x['wait_days'].median(),
            'H4 Spearman ρ': stats.spearmanr(tt['freq'], tt['maxdisc']).statistic,
            'H5 정확도 차이(최대-최소)': prec_.max() - prec_.min()}

m_steam_maxreg = metrics_table(prices_steam, 'max').loc[common]          # 정가를 "정가 중앙값" 대신 "1년 최고가"로 정의
rows = []
for label, t, (f_, d_) in (('중앙값 분할 (기본)', m_steam, thr['steam']), ('고정 기준(연 6회·70%)', m_steam, (6, 70)),
                           ('고정 기준(연 8회·60%)', m_steam, (8, 60)), ('고정 기준(연 10회·80%)', m_steam, (10, 80)),
                           ('정가 = 1년 최고가', m_steam_maxreg, thr['steam'])):
    r = type_stats(t, f_, d_)
    rows.append({'조건': label, **(r or {'가장 작은 유형(게임수)': f'유형이 너무 적음(<{MIN_TYPE_N})'})})
rob = pd.DataFrame(rows)
display(rob.round(3))
R['robustness'] = rob.round(4).to_dict('records')

ok_rows = rob.dropna(subset=['H1 구매가 차이(최대-최소)'])
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
y = np.arange(len(ok_rows))[::-1]
for k, (col, ref, title) in enumerate((('H1 구매가 차이(최대-최소)', PRACTICAL_PRICE, 'H1 구매가 차이(최대-최소)'),
                                       ('H2 상시-희소 구매가', -PRACTICAL_PRICE2, 'H2 상시-희소 구매가 (음수 = 상시가 더 쌈)'),
                                       ('H3 희소-상시 기다림(일)', PRACTICAL_WAIT, 'H3 희소-상시 기다리는 일수'))):
    ax[k].barh(y, ok_rows[col], color=BLUE); ax[k].axvline(ref, color=ORANGE, ls='--', label=f'실무 기준 {ref:g}'); ax[k].axvline(0, color='k', lw=.8)
    ax[k].set_title(title, fontsize=10); ax[k].legend(fontsize=8)
ax[0].set_yticks(y); ax[0].set_yticklabels(ok_rows['조건'])
savefig(fig, 'a5_5_robustness.png'); plt.show()

# 스토어 기준이 바뀌어도 같은 유형으로 분류되는가 (일치율과 Cohen's κ)
ct = pd.crosstab(m_steam['type'], m_all['type']).reindex(index=TYPES, columns=TYPES, fill_value=0)
n_ = ct.to_numpy().sum()
po = np.trace(ct.to_numpy()) / n_
pe = (ct.sum(axis=1).to_numpy() * ct.sum(axis=0).to_numpy()).sum() / n_ ** 2
kappa = float((po - pe) / (1 - pe))
print(f'Steam 기준과 전 스토어 기준에서 같은 유형으로 분류된 게임: {po:.1%} · Cohen\'s κ = {kappa:.2f} (0.4 미만이면 일치도가 낮음)')
display(ct.rename_axis('Steam 기준 ↓ / 전 스토어 기준 →'))
R['store_agreement'] = dict(agree=float(po), kappa=kappa)

### 차트 해설
분류 기준선(중앙값 / 고정 기준 3가지)과 정가 정의를 바꿔 가며 H1·H2·H3의 효과 크기를 반복한 것입니다. 주황 점선이 실무 기준입니다.

### 이 차트로 알 수 있는 것
- 모든 조건에서 막대가 주황 점선을 넘으면(H2는 점선보다 **왼쪽**) 결과가 분류 기준에 좌우되지 않는 **강건한 결과**입니다.
- 표 아래의 스토어 일치도가 낮으면(κ < 0.4) **같은 게임이 어느 스토어를 기준으로 보느냐에 따라 다른 유형이 된다**는 뜻입니다. 서비스에서 유형 라벨을 쓴다면 기준 스토어를 함께 밝혀야 합니다.

### 의미 판단
- **H1(구매가 차이)과 H2(상시 vs 희소)는 어떤 조건에서도 유지됩니다.** 분류 기준선 4가지와 정가 정의(정가 중앙값 → 1년 최고가)를 바꿔도 H1의 격차는 0.16~0.19(기준 0.1 이상), H2는 −0.06 ~ −0.12(기준 −0.05 이하)입니다.
- **H3(놓치면 기다리는 일수)는 기준선에 따라 흔들립니다.** 5.8~22.4일로 달라지고, "연 10회·80%" 기준에서는 5.8일로 실무 기준(7일)에 못 미칩니다. 근거가 약하다는 뜻입니다.
- **스토어 일치도는 낮습니다.** Steam 기준과 전 스토어 기준에서 같은 유형이 되는 게임은 50.3%이고 κ는 0.34입니다. 같은 게임이 어느 스토어를 기준으로 보느냐에 따라 다른 유형이 됩니다.
- 의미: 서비스에서 유형 라벨을 쓴다면 **기준 스토어를 함께 밝히고**, 라벨보다 "정가 대비 평균가", "할인 일수 비율" 같은 **숫자를 같이 보여 주는 것**이 안전합니다.

## 8. 결론 — 가설 대비 결과와 인사이트·제안

아래 문장은 위 분석 결과를 **그대로 불러와** 만들어집니다. 판정이 "불일치"인 가설은 데이터가 지지하지 않는다고 발표에서 솔직하게 말하고, 제안은 일치한 부분만 근거로 삼습니다.

In [ ]:
r1, r2, r3, r4, r5 = R['h1'], R['h2'], R['h3'], R['h4'], R['h5']
summary_tab = pd.DataFrame([
    ['H1 유형별 구매가가 다르다', f"{r1['spread']:.2f} 차이 (가장 싼 {min(r1['means'], key=r1['means'].get)} {min(r1['means'].values()):.2f})", f"p={r1['p']:.1e}", f"{r1['ci'][0]:.2f} ~ {r1['ci'][1]:.2f}", mark_r(r1)],
    ['H2 할인율 비슷해도 구매가가 다르다', f"할인율 {r2['disc_a']:.0f}% vs {r2['disc_b']:.0f}%, 구매가 {r2['price_a']:.2f} vs {r2['price_b']:.2f}", f"p={r2['p']:.1e}", f"{r2['ci'][0]:+.2f} ~ {r2['ci'][1]:+.2f}", mark_r(r2)],
    ['H3 희소 빅세일형이 더 오래 기다림', f"{r3['wait_b']:.1f}일 vs {r3['wait_a']:.1f}일", f"p={r3['p']:.1e}", f"{r3['ci'][0]:+.1f} ~ {r3['ci'][1]:+.1f}일", mark_r(r3)],
    ['H4 자주 할인할수록 얕다(미끼 할인)', f"ρ={r4['rho_steam']:+.2f}", f"p={r4['p_steam']:.2f}", f"{r4['ci'][0]:+.2f} ~ {r4['ci'][1]:+.2f}", mark_r(r4, 'p_steam')],
    ['H5 표시 정확도가 유형별로 다르다', f"차이 {r5['spread']:.2f}", f"p={r5['p']:.1e}", '-', mark_r(r5)],
], columns=['가설', '핵심 값', '검정', '95% 신뢰구간', '판정'])
summary_tab

In [ ]:
ok_rob = pd.DataFrame(R['robustness']).dropna(subset=['H1 구매가 차이(최대-최소)'])
h1_robust = bool((ok_rob['H1 구매가 차이(최대-최소)'] >= PRACTICAL_PRICE).all())
h2_robust = bool((ok_rob['H2 상시-희소 구매가'] <= -PRACTICAL_PRICE2).all())
h3_robust = bool((ok_rob['H3 희소-상시 기다림(일)'] >= PRACTICAL_WAIT).all())
sa = R['store_agreement']
L = []
L.append('■ 추가로 확인한 것 (위 판정표의 결과가 조건과 무관한지)')
L.append(f"  조건을 바꿔도(분류 기준선 4가지, 정가 정의) 결론 유지(H1은 격차 0.1 이상, H2는 -0.05 이하, H3은 7일 이상): H1={'예' if h1_robust else '아니오'}, H2={'예' if h2_robust else '아니오'}, H3={'예' if h3_robust else '아니오'}.")
L.append(f"  스토어 기준 일치도: 같은 유형으로 분류된 게임 {sa['agree']:.1%}, κ = {sa['kappa']:.2f}" + (' (낮음)' if sa['kappa'] < 0.4 else '.'))

L.append('\n■ 서비스 제안')
sh = R['share_half_sale']
L.append(f"  - 겜누리 최저가(전 스토어) 기준으로는 게임의 {sh['all']:.0f}%가 1년의 절반 넘게 할인 중이라(Steam만 {sh['steam']:.0f}%) \"할인 중\" 표시만으로는 좋은 딜을 가릴 수 없음.")
L.append('  - \'진짜 딜\' 표시는 할인율이 아니라 \'이 게임의 1년 가격 분포에서 지금 가격의 위치\'로 만들기. 예) 현재가가 1년 최저가의 105% 이내이고 할인 일수 비율이 낮은 게임.')
if r2['ok']:
    L.append(f"  - 최대 할인율이 같아도 실제 구매가가 다르므로(정가의 {r2['price_a'] * 100:.0f}% vs {r2['price_b'] * 100:.0f}%) 할인율 순위만으로 추천하지 않기.")
if r3['ok']:
    wmin, wmax = ok_rob['H3 희소-상시 기다림(일)'].min(), ok_rob['H3 희소-상시 기다림(일)'].max()
    L.append(f"  - 놓치면 다음 할인까지 평균 {r3['wait_b']:.0f}일 기다리는 희소 빅세일형 게임에는 목표가 알림 권유는 보조 제안으로만 둠 (상시 세일형과의 차이가 분류 기준선에 따라 {wmin:.1f}~{wmax:.1f}일로 달라져 근거가 약함).")
if not r4['ok']:
    L.append('  - \'퍼블리셔는 미끼 할인을 줄이라\'는 제안은 이 데이터로 뒷받침되지 않음 (H4 불일치, 그리고 이 분석은 구매 행동에 미치는 영향을 보지 못함).')
if sa['kappa'] < 0.4:
    L.append('  - 유형 라벨을 쓴다면 어느 스토어 기준인지 함께 표시하고, 라벨보다 \'정가 대비 평균가\', \'할인 일수 비율\' 같은 숫자를 같이 보여 주기.')

L.append('\n■ 데이터·파이프라인 시사점')
L.append(f"  - 분석 대상 {R['n_games']}개 게임(1년 이상 이력·유료). 같은 게임도 스토어를 섞은 최저가와 Steam 가격이 다른 이야기를 하므로, 서비스가 보여 주는 가격이 어느 스토어 기준인지가 분석과 화면 모두에서 중요함.")
L.append('  - price_history는 가격이 바뀐 때만 쌓이므로 일별 최저가는 스토어별 마지막 가격을 이어 채워 만들었다(서비스 timing.py와 같은 방식).')
print('\n'.join(L))

(RESULT_DIR / 'summary.json').write_text(json.dumps(R, ensure_ascii=False, indent=2, default=float), encoding='utf-8')
print('\n요약 저장:', RESULT_DIR / 'summary.json')

### 수익 연결 (아이디어 · 구매 데이터가 없어 효과는 검증하지 못함)
팀이 정한 수익 모델은 **제휴 수수료 · 광고 배너 · 메일 제휴 추천**입니다. 이 분석과는 이렇게 이어 볼 수 있습니다.
* **제휴 수수료**: 제휴로 돈을 버는 서비스는 사용자가 믿고 클릭해야 합니다. "진짜 딜"에만 표시를 붙이면 클릭 신뢰가 쌓이고, 이것이 장기적인 전환율의 기반입니다.
* **광고 배너·메일 추천**: "할인 중"이 아니라 **"이 게임의 1년 가격 분포에서 실제로 싼 딜"** 만 상단 노출이나 메일 추천에 올리는 기준으로 쓸 수 있습니다.
* 아직 구매 데이터가 없어서 실제 효과는 검증하지 못했습니다.

### 한계 (발표에서 먼저 밝힐 것)
| 한계 | 영향 | 대응 |
| --- | --- | --- |
| **인과가 아니라 관찰이다** | "잦은 할인이 정가 구매를 막는다"는 판매 데이터 없이는 검증할 수 없음 | 제안에서 퍼블리셔 전략 제안은 뺌. 해석은 "관찰된 가격 패턴" 수준으로 |
| 유형 정의가 곧 지표(빈도·깊이) | H1은 구조적으로 유의할 수밖에 없음 | p값 대신 효과 크기(정가 대비 %p)로 판정 |
| 유형은 연속적이고 중앙값 4분할은 관례 | "희소"도 Steam에서 연 7회 수준으로 절대적으로 드문 것은 아님. 이름은 상대적 표현 | 7장에서 분류 기준선 4가지로 확인, 라벨보다 숫자를 같이 제시 |
| **스토어 기준에 따라 유형이 바뀜** | 같은 게임이 두 기준에서 같은 유형일 확률이 절반 정도 | 기준 스토어를 명시. 겜누리 최저가 기준과 Steam 기준을 모두 계산 |
| 최근 **1년치 한 번**의 스냅샷, 인기작 위주 1,000개 | 연도가 바뀌면 달라질 수 있고 전체 게임으로 일반화 불가 | 결과 날짜와 표본 수를 함께 표기, 이력이 쌓이면 다시 실행 |
| 같은 세일에 많은 게임이 같이 들어감 | 게임 간 독립이 아니라 p값이 실제보다 작게 나옴 | 판정은 효과 크기 우선, 신뢰구간 병기 |
| 정가는 정가 중앙값으로 정의, 정가가 바뀐 게임은 왜곡 가능 | 평균 구매가가 달라짐 | 7장에서 "정가 = 1년 최고가"로 바꿔 확인 |